In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from metrics.fix_coords import fix_coords_by_svd, fix_coords_by_xyz
from metrics.gdt_ts import gdt_ts
from metrics.tmscore import tmsocre
from metrics.lddt import lddt
from metrics.rmsd import rmsd


In [34]:
base = np.load('./full/results.npz', allow_pickle=True)
noboth = np.load('./no_both/results.npz', allow_pickle=True)
noQCSE = np.load('./no_QCSE/results.npz', allow_pickle=True)
noQIRE = np.load('./no_QIRE/results.npz', allow_pickle=True)
print(len(base.files), len(noboth.files), len(noQCSE.files), len(noQIRE.files))

35 35 35 35


In [35]:
all_pdbs = list(base.files)
references = {}
for pdb in all_pdbs:
    references[pdb] = np.load(f'../../../data/FormatData/{pdb}.npz', allow_pickle=True)['coord']

In [50]:
result = {
    'pdb_id': [],
    'base_rmsd': [], 'base_gdt_ts': [], 'base_tmscore': [], 'base_lddt': [],
    'noboth_rmsd': [], 'noboth_gdt_ts': [], 'noboth_tmscore': [], 'noboth_lddt': [],
    'noQCSE_rmsd': [], 'noQCSE_gdt_ts': [], 'noQCSE_tmscore': [], 'noQCSE_lddt': [],
    'noQIRE_rmsd': [], 'noQIRE_gdt_ts': [], 'noQIRE_tmscore': [], 'noQIRE_lddt': [],
}
result = {
    'svd': result,
    'f3kp': result,
}

for align in ['svd', 'f3kp']:
    for pdb_id in all_pdbs:
        result[align]['pdb_id'].append(pdb_id)
        
        reference = fix_coords_by_xyz(references[pdb_id])
        base_coords = fix_coords_by_xyz(base[pdb_id].mean(axis=0))
        boboth_coords = fix_coords_by_xyz(noboth[pdb_id].mean(axis=0))
        noQCSE_coords = fix_coords_by_xyz(noQCSE[pdb_id].mean(axis=0))
        noQIRE_coords = fix_coords_by_xyz(noQIRE[pdb_id].mean(axis=0))
        
        if align == 'svd':
            
            # speical process for symetric matrix
            if boboth_coords[3, 0] == 0:
                boboth_coords = boboth_coords + 1e-39
            if noQCSE_coords[3, 0] == 0:
                noQCSE_coords = noQCSE_coords + 1e-39
            if noQIRE_coords[3, 0] == 0:
                noQIRE_coords = noQIRE_coords + 1e-39
            
            base_coords = fix_coords_by_svd(reference, base_coords)
            boboth_coords = fix_coords_by_svd(reference, boboth_coords)
            noQCSE_coords = fix_coords_by_svd(reference, noQCSE_coords)
            noQIRE_coords = fix_coords_by_svd(reference, noQIRE_coords)
        
        result[align]['base_rmsd'].append(rmsd(reference, base_coords))
        result[align]['base_gdt_ts'].append(gdt_ts(reference, base_coords))
        result[align]['base_tmscore'].append(tmsocre(reference, base_coords))
        result[align]['base_lddt'].append(lddt(reference, base_coords))

        result[align]['noboth_rmsd'].append(rmsd(reference, boboth_coords))
        result[align]['noboth_gdt_ts'].append(gdt_ts(reference, boboth_coords))
        result[align]['noboth_tmscore'].append(tmsocre(reference, boboth_coords))
        result[align]['noboth_lddt'].append(lddt(reference, boboth_coords))
        
        result[align]['noQCSE_rmsd'].append(rmsd(reference, noQCSE_coords))
        result[align]['noQCSE_gdt_ts'].append(gdt_ts(reference, noQCSE_coords))
        result[align]['noQCSE_tmscore'].append(tmsocre(reference, noQCSE_coords))
        result[align]['noQCSE_lddt'].append(lddt(reference, noQCSE_coords))
        
        result[align]['noQIRE_rmsd'].append(rmsd(reference, noQIRE_coords))
        result[align]['noQIRE_gdt_ts'].append(gdt_ts(reference, noQIRE_coords))
        result[align]['noQIRE_tmscore'].append(tmsocre(reference, noQIRE_coords))
        result[align]['noQIRE_lddt'].append(lddt(reference, noQIRE_coords))
        
        # if pdb_id=='7n2i':
        #     print(f'{pdb_id} done {align}')
        #     print('QCSE: ', result[align]['noQCSE_rmsd'][-1], result[align]['noQCSE_gdt_ts'][-1], result[align]['noQCSE_tmscore'][-1], result[align]['noQCSE_lddt'][-1])
        #     print('noboth: ', result[align]['noboth_rmsd'][-1], result[align]['noboth_gdt_ts'][-1], result[align]['noboth_tmscore'][-1], result[align]['noboth_lddt'][-1])
        #     print(noQCSE_coords, boboth_coords)
            
    result[align] = pd.DataFrame(result[align])
    result[align].to_csv(f'./{align}_results.csv', index=False)


In [51]:
print(
    'align: svd', '\n', 
    'base_rmsd: ', np.mean(result['svd']['base_rmsd']), '\n',
    'base_gdt_ts: ', np.mean(result['svd']['base_gdt_ts']), '\n',
    'base_tmscore: ', np.mean(result['svd']['base_tmscore']), '\n',
    'base_lddt: ', np.mean(result['svd']['base_lddt']), '\n',
    'noboth_rmsd: ', np.mean(result['svd']['noboth_rmsd']), '\n',
    'noboth_gdt_ts: ', np.mean(result['svd']['noboth_gdt_ts']), '\n',
    'noboth_tmscore: ', np.mean(result['svd']['noboth_tmscore']), '\n',
    'noboth_lddt: ', np.mean(result['svd']['noboth_lddt']), '\n',
    'noQCSE_rmsd: ', np.mean(result['svd']['noQCSE_rmsd']), '\n',
    'noQCSE_gdt_ts: ', np.mean(result['svd']['noQCSE_gdt_ts']), '\n',
    'noQCSE_tmscore: ', np.mean(result['svd']['noQCSE_tmscore']), '\n',
    'noQCSE_lddt: ', np.mean(result['svd']['noQCSE_lddt']), '\n',
    'noQIRE_rmsd: ', np.mean(result['svd']['noQIRE_rmsd']), '\n',
    'noQIRE_gdt_ts: ', np.mean(result['svd']['noQIRE_gdt_ts']), '\n',
    'noQIRE_tmscore: ', np.mean(result['svd']['noQIRE_tmscore']), '\n',
    'noQIRE_lddt: ', np.mean(result['svd']['noQIRE_lddt']), '\n', 
)

print(
    'align: f3kp', '\n', 
    'base_rmsd: ', np.mean(result['f3kp']['base_rmsd']), '\n',
    'base_gdt_ts: ', np.mean(result['f3kp']['base_gdt_ts']), '\n',
    'base_tmscore: ', np.mean(result['f3kp']['base_tmscore']), '\n',
    'base_lddt: ', np.mean(result['f3kp']['base_lddt']), '\n',
    'noboth_rmsd: ', np.mean(result['f3kp']['noboth_rmsd']), '\n',
    'noboth_gdt_ts: ', np.mean(result['f3kp']['noboth_gdt_ts']), '\n',
    'noboth_tmscore: ', np.mean(result['f3kp']['noboth_tmscore']), '\n',
    'noboth_lddt: ', np.mean(result['f3kp']['noboth_lddt']), '\n',
    'noQCSE_rmsd: ', np.mean(result['f3kp']['noQCSE_rmsd']), '\n',
    'noQCSE_gdt_ts: ', np.mean(result['f3kp']['noQCSE_gdt_ts']), '\n',
    'noQCSE_tmscore: ', np.mean(result['f3kp']['noQCSE_tmscore']), '\n',
    'noQCSE_lddt: ', np.mean(result['f3kp']['noQCSE_lddt']), '\n',
    'noQIRE_rmsd: ', np.mean(result['f3kp']['noQIRE_rmsd']), '\n',
    'noQIRE_gdt_ts: ', np.mean(result['f3kp']['noQIRE_gdt_ts']), '\n',
    'noQIRE_tmscore: ', np.mean(result['f3kp']['noQIRE_tmscore']), '\n',
    'noQIRE_lddt: ', np.mean(result['f3kp']['noQIRE_lddt']), '\n', 
)
    

align: svd 
 base_rmsd:  5.174472269543991 
 base_gdt_ts:  0.48644557823129253 
 base_tmscore:  0.3619605508925762 
 base_lddt:  0.6653869992441421 
 noboth_rmsd:  8.344101889542136 
 noboth_gdt_ts:  0.39623866213151926 
 noboth_tmscore:  0.29771800038127066 
 noboth_lddt:  0.6859967876039305 
 noQCSE_rmsd:  8.760687318227676 
 noQCSE_gdt_ts:  0.3535232426303854 
 noQCSE_tmscore:  0.24925944119047902 
 noQCSE_lddt:  0.6859967876039305 
 noQIRE_rmsd:  5.520030579999603 
 noQIRE_gdt_ts:  0.454047619047619 
 noQIRE_tmscore:  0.3279170659529249 
 noQIRE_lddt:  0.7061581632653061 

align: f3kp 
 base_rmsd:  6.679322757581163 
 base_gdt_ts:  0.47099773242630383 
 base_tmscore:  0.38446424244922656 
 base_lddt:  0.6653869992441421 
 noboth_rmsd:  8.080390202783624 
 noboth_gdt_ts:  0.4174801587301587 
 noboth_tmscore:  0.3288722515990593 
 noboth_lddt:  0.6859967876039303 
 noQCSE_rmsd:  8.288683171954652 
 noQCSE_gdt_ts:  0.39612244897959187 
 noQCSE_tmscore:  0.30464296892801207 
 noQCSE_ld

In [52]:

import pandas as pd
import numpy as np

f3kp = pd.read_csv('./f3kp_results.csv')
svd = pd.read_csv('./svd_results.csv')
result = {
    'svd': svd, 'f3kp': f3kp
}

df = {'metrics': ['GDT-TS', 'lDDT', 'TM-score', 'RMSD']}
for case in ['base', 'noboth', 'noQCSE', 'noQIRE']:
    for align in ['svd', 'f3kp']:
        name = f'{case}-{align}'
        df[name] = []
        df[name].append(np.mean(result[align][f'{case}_gdt_ts']))
        df[name].append(np.mean(result[align][f'{case}_lddt']))
        df[name].append(np.mean(result[align][f'{case}_tmscore']))
        df[name].append(np.mean(result[align][f'{case}_rmsd']))            
df = pd.DataFrame(df)
df.to_csv('./results.csv', index=False)